# 02. QLoRA fine-tuning

Fine-tuning `Qwen2.5-7B-Instruct` (same 4-bit checkpoint as in `01_icl`) with LoRA adapters on the **whole training set**.
All choices are made on **`rand-dev`**, changing **one factor at a time** (see `docs/lora_notes.md`):

1. where to add LoRA: attention queries and values only (`qv`) vs every linear layer (`all`);
2. decoding of the fine-tuned generative model: free generation vs label log-probability (no extra training);
3. generating the label vs a classification head on the last token;
4. with vs without the lemma.

The best configuration is run once on `rand-test` and `freq-test` at the end of the notebook.
Runtime: GPU (T4). On Kaggle, use *Run All* in an interactive session (background versions can wait in a queue for
hours); on Colab, accept the Google Drive prompt at the start. Each training takes one epoch over 10,604 senses;
trained adapters are saved (Google Drive on Colab), so a restarted session reuses them.

In [ ]:
# Setup: on Colab or Kaggle, clone (or update) the repo so that `src/` is importable; locally, run from the repo root.
import os, sys
REPO_URL = "https://github.com/bkadyrbay-hash/lab1-supersense-llm.git"
BRANCH = "b/lora"  # set to your own branch to test code that is not merged yet
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if "google.colab" in sys.modules or ON_KAGGLE:
    REPO_DIR = ("/kaggle/working" if ON_KAGGLE else "/content") + "/lab1-supersense-llm"
    if not os.path.exists(REPO_DIR):
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
!pip install -q bitsandbytes peft  # the rest is preinstalled in Colab and Kaggle

In [ ]:
# All hyperparameters of this notebook.
CONFIG = {
    "model": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # same 4-bit checkpoint as 01_icl
    "tuning_split": "rand-dev",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "epochs": 1,
    "batch_size": 4,
    "grad_accum": 4,  # effective batch of 16 sequences
    "learning_rate": 2e-4,
    "warmup_share": 0.03,
    "eval_batch_size": 32,
    "max_new_tokens": 10,
    "seed": 42,
}
TARGET_MODULES = {
    "qv": ["q_proj", "v_proj"],
    "all": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}
AUTHOR = ("Alena Muravyeva", "90255314+Alhassliebe@users.noreply.github.com")  # author of the result commits

In [ ]:
import gc
import math
import time
from pathlib import Path

import pandas as pd
import torch
import transformers
from peft import LoraConfig, PeftModel, get_peft_model
from tqdm.auto import tqdm
from transformers import (AutoConfig, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorForSeq2Seq, DataCollatorWithPadding, Trainer,
                          TrainingArguments)

from src.data import HYPERSENSES, SUPER2HYPER, SUPERSENSES, format_input, get_split, load_data
from src.evaluate import RESULTS_DIR, evaluate, log_experiment, paired_bootstrap, save_predictions
from src.label_scoring import label_token_ids, score_labels
from src.prompts import SOURCES, parse_answer
from src.sync import github_token, push_results

transformers.logging.set_verbosity_error()
GPU = torch.cuda.is_available()
DTYPE = torch.float16 if GPU else torch.float32  # the T4 has no bfloat16; on CPU, fp16 training diverges
df = load_data()
train = get_split(df, "train")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
labels_ids = label_token_ids(tokenizer, "en")  # answer tokens of each label: its name, then <|im_end|>
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER_DIR = Path("/content/drive/MyDrive/lab1_lora_adapters")
else:
    ADAPTER_DIR = Path("outputs/lora_adapters")  # on Kaggle, kept as an output of the notebook
# False means the GH_TOKEN secret is missing: results then exist only in this session and are lost with it.
print("Results pushed to GitHub after each run:", github_token() is not None)

## Inputs

The prompt is a short English instruction (English worked better than French for ICL, see `01_icl`) and the sense;
the 25 labels and their glosses are not listed: after fine-tuning the model knows the label set, and the prompt is
about 75 tokens instead of about 500 for the 0-shot ICL prompt, which makes training much cheaper. With the **generation** head the model learns to write the label followed by
the end-of-turn token, and the loss is computed on these answer tokens only. With the **classification** head a new
linear layer maps the last prompt token to the 25 supersenses.

In [ ]:
SYSTEM = "Classify the sense of a French noun, described by {source}, into its supersense. Answer with the supersense only."


def prompt_ids(senses, with_lemma):
    """Token ids of the chat up to the start of the assistant's answer, one list per sense."""
    system = SYSTEM.format(source=SOURCES["en"][with_lemma])
    texts = [tokenizer.apply_chat_template([{"role": "system", "content": system},
                                            {"role": "user", "content": format_input(row, with_lemma)}],
                                           add_generation_prompt=True, tokenize=False)
             for _, row in senses.iterrows()]
    return tokenizer(texts, add_special_tokens=False).input_ids


def training_examples(head, with_lemma):
    """Tokenized training set; -100 masks the prompt out of the generation loss."""
    label_idx = train["supersense"].map(SUPERSENSES.index)
    prompts = prompt_ids(train, with_lemma)
    if head == "generation":
        return [{"input_ids": p + labels_ids[i], "labels": [-100] * len(p) + labels_ids[i]}
                for p, i in zip(prompts, label_idx)]
    return [{"input_ids": p, "labels": i} for p, i in zip(prompts, label_idx)]

## Training and prediction

One 7B model fits on a T4, not two: the previous model is freed before the next one is loaded. Adapters are saved
under `ADAPTER_DIR/<configuration>`, so a configuration is trained only once, even across sessions.

In [ ]:
loaded = {}


def adapter_name(targets, head, with_lemma):
    """Name of a trained configuration, e.g. qv_generation_lemma."""
    return f"{targets}_{head}_{'lemma' if with_lemma else 'nolemma'}"


def load_base(head):
    """4-bit base model with its language-modelling head, or with a new 25-class head ('score')."""
    model_class, extra = ((AutoModelForCausalLM, {}) if head == "generation" else
                          (AutoModelForSequenceClassification,
                           {"num_labels": len(SUPERSENSES), "pad_token_id": tokenizer.pad_token_id}))
    config = AutoConfig.from_pretrained(CONFIG["model"], **extra)
    quantization = getattr(config, "quantization_config", None)
    if quantization:  # the checkpoint quantizes every layer it does not list: 'score' must stay 16-bit to be trained
        quantization["llm_int8_skip_modules"] = quantization["llm_int8_skip_modules"] + ["score"]
    # About 6 GB in 4-bit: the whole model goes on one GPU (on Kaggle's T4 x2, splitting it brings no benefit).
    model = model_class.from_pretrained(CONFIG["model"], config=config, dtype=DTYPE,
                                        device_map={"": 0} if GPU else None)
    if head == "classification":
        # Zero init: training starts from uniform predictions (loss ln 25 = 3.2); with a random init, the large final
        # hidden states of Qwen give huge logits (initial loss about 30 in a test with Qwen2.5-0.5B).
        torch.nn.init.zeros_(model.score.weight)
    for module in model.modules():  # the checkpoint computes in bfloat16, which the T4 does not support
        if hasattr(module, "compute_dtype"):
            module.compute_dtype = torch.float16
    return model


def get_model(targets, head, with_lemma):
    """Fine-tuned model of one configuration: its saved adapter if it exists, otherwise trained now and saved."""
    name = adapter_name(targets, head, with_lemma)
    if loaded.get("name") == name:
        return loaded["model"]
    loaded.clear()
    gc.collect()
    torch.cuda.empty_cache()
    transformers.set_seed(CONFIG["seed"])  # LoRA matrices and the classification head are initialised randomly
    base = load_base(head)
    if (ADAPTER_DIR / name).exists():
        model = PeftModel.from_pretrained(base, ADAPTER_DIR / name)
    else:
        lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
                          target_modules=TARGET_MODULES[targets],
                          task_type="CAUSAL_LM" if head == "generation" else "SEQ_CLS")  # SEQ_CLS also trains 'score'
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model = get_peft_model(base, lora)
        # fp16 mixed precision needs fp32 trained weights; the frozen embeddings and lm_head stay in fp16:
        # in fp32 they would take 2.2 GB more on the T4.
        for param in model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
        model.print_trainable_parameters()
        examples = training_examples(head, with_lemma)
        steps = math.ceil(len(examples) / (CONFIG["batch_size"] * CONFIG["grad_accum"])) * CONFIG["epochs"]
        args = TrainingArguments(
            output_dir="outputs/trainer", num_train_epochs=CONFIG["epochs"],
            per_device_train_batch_size=CONFIG["batch_size"], gradient_accumulation_steps=CONFIG["grad_accum"],
            learning_rate=CONFIG["learning_rate"],
            lr_scheduler_type="cosine", warmup_steps=int(CONFIG["warmup_share"] * steps),
            fp16=GPU, logging_steps=50, save_strategy="no", report_to="none",
            disable_tqdm=False,  # keep the progress bar although transformers warnings are silenced
            seed=CONFIG["seed"],
        )
        collator = DataCollatorForSeq2Seq(tokenizer) if head == "generation" else DataCollatorWithPadding(tokenizer)
        trainer = Trainer(model=model, args=args, train_dataset=examples, data_collator=collator)
        trainer.train()
        # accelerate keeps the optimizer and model references until told otherwise; without this, the next
        # configuration is loaded while this one is still on the GPU and runs out of memory.
        trainer.accelerator.free_memory()
        model.save_pretrained(ADAPTER_DIR / name)
    model.eval()
    loaded.update(name=name, model=model)
    return model


@torch.no_grad()
def predict(model, prompts, head, decoding):
    """Predicted supersense and raw output (generated text, or the 3 best labels with their log-prob) per sense."""
    preds, raws = [], []
    if decoding == "logprob":  # one prompt at a time: score_labels scores the 25 labels as a batch
        for ids in tqdm(prompts):
            scores = score_labels(model, torch.tensor([ids], device=model.device), labels_ids)
            ranked = sorted(zip(scores, SUPERSENSES), reverse=True)
            raws.append(" | ".join(f"{label} {score:.2f}" for score, label in ranked[:3]))
            preds.append(ranked[0][1])
        return preds, raws
    for start in tqdm(range(0, len(prompts), CONFIG["eval_batch_size"])):
        # Left padding so that generation continues each prompt; right padding matches training for the classifier.
        batch = tokenizer.pad({"input_ids": prompts[start:start + CONFIG["eval_batch_size"]]}, return_tensors="pt",
                              padding_side="left" if head == "generation" else "right").to(model.device)
        if head == "classification":
            top = model(**batch).logits.float().log_softmax(-1).topk(3)
            preds += [SUPERSENSES[i] for i in top.indices[:, 0].tolist()]
            raws += [" | ".join(f"{SUPERSENSES[i]} {s:.2f}" for s, i in zip(row_s, row_i))
                     for row_s, row_i in zip(top.values.tolist(), top.indices.tolist())]
        else:
            output = model.generate(**batch, max_new_tokens=CONFIG["max_new_tokens"], do_sample=False,
                                    temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.pad_token_id)
            texts = tokenizer.batch_decode(output[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
            raws += texts
            preds += [parse_answer(text, "en") for text in texts]
    return preds, raws


def run(targets, head="generation", with_lemma=True, decoding="generation", split=CONFIG["tuning_split"]):
    """Train (or reload) one configuration, predict `split`, save and log; reuse saved predictions if already run."""
    decoding = decoding if head == "generation" else "argmax"
    name = adapter_name(targets, head, with_lemma)
    cid = name + ("_logprob" if decoding == "logprob" else "")
    path = RESULTS_DIR / "predictions" / f"lora__{cid}__{split}.csv"
    fresh = not path.exists()
    if fresh:
        trained_now = not (ADAPTER_DIR / name).exists()
        start = time.time()
        eval_df = get_split(df, split)
        model = get_model(targets, head, with_lemma)
        # Same mixed precision as in training: fp32 trained weights ('score') applied to fp16 activations.
        with torch.autocast(model.device.type, dtype=DTYPE, enabled=GPU):
            pred_labels, raws = predict(model, prompt_ids(eval_df, with_lemma), head, decoding)
        runtime = (time.time() - start) / 60
        preds = eval_df[["sense_id", "lemma", "definition"]].assign(
            gold_supersense=eval_df["supersense"], pred_supersense=pred_labels,
            gold_hypersense=eval_df["hypersense"], raw_output=raws)
        preds.insert(6, "pred_hypersense", preds["pred_supersense"].map(SUPER2HYPER))
        save_predictions(preds, "lora", cid, split)
    else:  # Colab sessions are cut regularly: never pay twice for the same run
        preds = pd.read_csv(path, keep_default_na=False).replace("", None)
    sup = evaluate(preds["gold_supersense"].tolist(), preds["pred_supersense"].tolist(), SUPERSENSES)
    hyp = evaluate(preds["gold_hypersense"].tolist(), preds["pred_hypersense"].tolist(), HYPERSENSES)
    if fresh:
        log_experiment({"author": "B", "method": "lora", "config_id": cid, "split": split, "model": CONFIG["model"],
                        "with_lemma": with_lemma, "prompt_lang": "en", "lora_targets": targets, "head": head,
                        "decoding": decoding, "epochs": CONFIG["epochs"],
                        "accuracy_super": round(sup["accuracy"], 4), "macro_f1_super": round(sup["macro_f1"], 4),
                        "accuracy_hyper": round(hyp["accuracy"], 4), "macro_f1_hyper": round(hyp["macro_f1"], 4),
                        "invalid_rate": round(sup["invalid_rate"], 4), "n": sup["n"], "runtime_min": round(runtime, 1),
                        "notes": "runtime includes training" if trained_now else ""})
        push_results(f"Add LoRA run {cid} on {split}", BRANCH, AUTHOR)
    print(f"{cid:35s} macro-F1 {sup['macro_f1']:.3f}  acc {sup['accuracy']:.3f}  "
          f"hyper macro-F1 {hyp['macro_f1']:.3f}  invalid {sup['invalid_rate']:.1%}")
    return {"config_id": cid, "macro_f1": sup["macro_f1"], "preds": preds}


def compare(a, b):
    """Print the share of bootstrap resamples where b has a higher macro-F1 than a.

    Close to 100 % or 0 % means a consistent difference, around 50 % means none.
    """
    share = paired_bootstrap(a["preds"]["gold_supersense"], a["preds"]["pred_supersense"],
                             b["preds"]["pred_supersense"], SUPERSENSES)
    print(f"{b['config_id']} beats {a['config_id']} in {share:.0%} of bootstrap resamples")

Each section keeps the setting with the higher macro-F1; a tie goes to the first, simpler one
(`qv`, generation, with lemma), since `max` returns the first maximum.

## 1. Where to add LoRA

`qv`: queries and values of the attention layers, as in the original LoRA paper (Hu et al., 2022);
`all`: every linear layer of attention and MLP, as recommended for QLoRA (Dettmers et al., 2023).
Generation head, with the lemma.

In [ ]:
target_runs = {t: run(t) for t in TARGET_MODULES}
compare(target_runs["qv"], target_runs["all"])
TARGETS = max(target_runs, key=lambda t: target_runs[t]["macro_f1"])
print("LoRA modules kept:", TARGETS)

## 2. Generation vs label log-probability

Same adapter: the fine-tuned model either generates its answer, or scores the 25 possible answers
(`src/label_scoring.py`, as in `01_icl`).

In [ ]:
decoding_runs = {"generation": target_runs[TARGETS], "logprob": run(TARGETS, decoding="logprob")}
compare(decoding_runs["generation"], decoding_runs["logprob"])
DECODING = max(decoding_runs, key=lambda d: decoding_runs[d]["macro_f1"])
print("Decoding kept:", DECODING)

## 3. Generation vs classification head

In [ ]:
head_runs = {"generation": decoding_runs[DECODING], "classification": run(TARGETS, "classification")}
compare(head_runs["generation"], head_runs["classification"])
HEAD = max(head_runs, key=lambda h: head_runs[h]["macro_f1"])
print("Head kept:", HEAD)

## 4. Is the lemma useful?

In [ ]:
lemma_runs = {True: head_runs[HEAD], False: run(TARGETS, HEAD, with_lemma=False, decoding=DECODING)}
compare(lemma_runs[False], lemma_runs[True])
WITH_LEMMA = lemma_runs[True]["macro_f1"] >= lemma_runs[False]["macro_f1"]
print("Lemma kept:", WITH_LEMMA)

## Summary of the tuning runs

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
metrics[(metrics["method"] == "lora") & (metrics["split"] == CONFIG["tuning_split"])].drop_duplicates(
    "config_id", keep="last").sort_values("macro_f1_super", ascending=False)[
    ["config_id", "macro_f1_super", "accuracy_super", "macro_f1_hyper", "invalid_rate", "runtime_min"]]

## Final evaluation on the test sets

The configuration chosen on `rand-dev` above is run **once** on each test set; nothing is tuned after this point.

In [ ]:
test_runs = {split: run(TARGETS, HEAD, WITH_LEMMA, DECODING, split=split) for split in ["rand-test", "freq-test"]}

## Download the results

Without a `GH_TOKEN` secret, the prediction files and log rows are zipped and downloaded from Colab, then committed from
a computer. On Kaggle, everything written under `/kaggle/working` is kept as the notebook's output.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    !cd /content/lab1-supersense-llm && zip -qr /content/lora_results.zip results
    files.download("/content/lora_results.zip")